# Aspectos preliminares

Este script tiene por objetivo verificar el contenido de los datasets luego de la normalización.

In [1]:
# Importacion de librerias.
import pandas as pd
import numpy as np
import os
from pandasql import sqldf

In [2]:
# Inicializacion de variables.
lookup_tables = ['application_train', 'bureau', 'previous_application']
lookup_dir = '../output/'

# Ejecución del control

### Application_train

In [3]:
# Función para cargar DataFrames desde archivos CSV en una carpeta específica
def load_dataframes(folder_path, table_name):
    """
    Carga DataFrames desde archivos CSV en la carpeta especificada.
    
    Args:
        folder_path (str): Ruta a la carpeta base (ej. '../output/').
        table_name (str): Nombre de la subcarpeta (ej. 'application_train').
    
    Returns:
        dict: Diccionario con nombres de archivos (sin extensión) como claves y DataFrames como valores.
    """
    full_path = os.path.join(folder_path, table_name)
    
    # Verificar si la carpeta existe
    if not os.path.exists(full_path):
        print(f"La carpeta {full_path} no existe.")
        return {}
    
    # Listar archivos CSV en la carpeta
    csv_files = [f for f in os.listdir(full_path) if f.endswith('.csv')]
    
    if not csv_files:
        print("No se encontraron archivos CSV en la carpeta.")
        return {}
    
    # Diccionario para almacenar los DataFrames
    dataframes = {}
    
    # Leer cada archivo CSV y crear un DataFrame
    for file in csv_files:
        file_path = os.path.join(full_path, file)
        try:
            df = pd.read_csv(file_path)
            # Usar nombre del archivo sin extensión como clave
            key = file.replace('.csv', '')
            dataframes[key] = df
            print(f"DataFrame creado para '{key}' con shape {df.shape}.")
        except Exception as e:
            print(f"Error al leer '{file}': {e}")
    
    # Opcional: Mostrar un resumen de los DataFrames creados
    print("\nResumen de DataFrames creados:")
    for name, df in dataframes.items():
        print(f"- {name}: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    return dataframes

In [4]:
# Llamada a la función para cargar los DataFrames
application_train_dataframes = load_dataframes(lookup_dir, lookup_tables[0])

DataFrame creado para 'application_train' con shape (307511, 123).
DataFrame creado para 'dim_code_gender' con shape (3, 2).
DataFrame creado para 'dim_emergencystate_mode' con shape (2, 2).
DataFrame creado para 'dim_flag_own_car' con shape (2, 2).
DataFrame creado para 'dim_flag_own_realty' con shape (2, 2).
DataFrame creado para 'dim_fondkapremont_mode' con shape (4, 2).
DataFrame creado para 'dim_housetype_mode' con shape (3, 2).
DataFrame creado para 'dim_name_contract_type' con shape (2, 2).
DataFrame creado para 'dim_name_education_type' con shape (5, 2).
DataFrame creado para 'dim_name_family_status' con shape (6, 2).
DataFrame creado para 'dim_name_housing_type' con shape (6, 2).
DataFrame creado para 'dim_name_income_type' con shape (8, 2).
DataFrame creado para 'dim_name_type_suite' con shape (7, 2).
DataFrame creado para 'dim_occupation_type' con shape (18, 2).
DataFrame creado para 'dim_organization_type' con shape (35, 2).
DataFrame creado para 'dim_organization_type_2' c

In [5]:
# Head de muestra.
application_train_dataframes['application_train'].head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE_ID,CODE_GENDER_ID,FLAG_OWN_CAR_ID,FLAG_OWN_REALTY_ID,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,ORGANIZATION_TYPE_2_ID
0,100002,1,1,1,1,1,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,100003,0,1,2,1,2,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,100004,0,2,1,2,1,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2
3,100006,0,1,2,1,1,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,1
4,100007,0,1,1,1,1,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2


In [6]:
# Query diseñada para unir application_train con las tablas de dimensión, reemplazando IDs por descripciones

# Identificar la tabla principal y las tablas de dimensión
main_table = 'application_train'
dim_tables = {k: v for k, v in application_train_dataframes.items() if k.startswith('dim_')}

# Obtener columnas de la tabla principal
main_columns = list(application_train_dataframes[main_table].columns)

# Construir la lista de columnas para SELECT, reemplazando IDs por descripciones
select_columns = []
for col in main_columns:
    if col.endswith('_ID'):
        # Encontrar la tabla de dimensión correspondiente
        dim_base = col.replace('_ID', '').lower()
        dim_name = f"dim_{dim_base}"
        if dim_name in dim_tables:
            # Usar la segunda columna (descripción) de la dim table
            desc_col = application_train_dataframes[dim_name].columns[1]
            select_columns.append(f"{dim_name}.{desc_col} AS {desc_col}")
        else:
            select_columns.append(f"{main_table}.{col}")
    else:
        select_columns.append(f"{main_table}.{col}")

# Construir la query SQL
query = f"SELECT {', '.join(select_columns)} FROM {main_table}"

for dim_name in dim_tables:
    # Asumir que el ID es la primera columna de cada tabla de dimensión
    id_col = application_train_dataframes[dim_name].columns[0]
    query += f" LEFT JOIN {dim_name} ON {main_table}.{id_col} = {dim_name}.{id_col}"

# Ejecutar la query usando pandasql
result = sqldf(query, application_train_dataframes)

# Mostrar el resultado (primeras filas para comparación)
result.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,ORGANIZATION_TYPE_2
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,Type 3
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Type 3
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0


In [7]:
# Lectura del df original para comparación.
original_df = pd.read_csv('../data/application_train.csv')

In [8]:
# Control de shape.
print("Shape original:", original_df.shape)
print("Shape resultante:", result.shape)

Shape original: (307511, 122)
Shape resultante: (307511, 123)


In [9]:
# Verificación de columnas en original_df.
for col in original_df.columns:
    if col in result.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

# Verificación de columnas en result.
for col in result.columns:
    if col in original_df.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

Columna 'ORGANIZATION_TYPE_2' NO está presente en el resultado.


In [10]:
# Script para comparar el contenido columna por columna entre original_df y result, excluyendo ORGANIZATION_TYPE y ORGANIZATION_TYPE_2

# Columnas a excluir
exclude_columns = ['ORGANIZATION_TYPE', 'ORGANIZATION_TYPE_2']

# Obtener columnas comunes, excluyendo las especificadas
common_columns = [col for col in original_df.columns if col not in exclude_columns]

print("Comparación columna por columna entre original_df y result:\n")

all_match = True

for col in common_columns:
    if col in result.columns:
        # Verificar si es una columna de ID (termina con '_ID')
        if col.endswith('_ID'):
            # Asumir que está correctamente reemplazada, no marcar como no coincidente
            continue
        else:
            # Comparar valores directamente
            if not original_df[col].equals(result[col]):
                print(f"Columna '{col}' no coincide.")
                all_match = False
    else:
        print(f"Columna '{col}' no está presente en result.")
        all_match = False

if all_match:
    print("Todas las columnas coinciden.")

print("\nComparación completada.")

Comparación columna por columna entre original_df y result:

Todas las columnas coinciden.

Comparación completada.


In [11]:
# Detalle de diferencias encontradas.
for col in original_df.columns:
    if col == 'ORGANIZATION_TYPE' or col == 'ORGANIZATION_TYPE_2':
        continue
    else:   
        differences = sum(original_df[col] != result[col])
        if differences > 0:
            print(f"Columna '{col}' tiene {differences} diferencias.", end=' ')
            print(original_df[original_df[col] != result[col]][col].value_counts(dropna=False).index[0])
        else:
            continue

Columna 'AMT_ANNUITY' tiene 12 diferencias. nan
Columna 'AMT_GOODS_PRICE' tiene 278 diferencias. nan
Columna 'NAME_TYPE_SUITE' tiene 1292 diferencias. nan
Columna 'OWN_CAR_AGE' tiene 202929 diferencias. nan
Columna 'OCCUPATION_TYPE' tiene 96391 diferencias. nan
Columna 'CNT_FAM_MEMBERS' tiene 2 diferencias. nan
Columna 'EXT_SOURCE_1' tiene 173378 diferencias. nan
Columna 'EXT_SOURCE_2' tiene 660 diferencias. nan
Columna 'EXT_SOURCE_3' tiene 60965 diferencias. nan
Columna 'APARTMENTS_AVG' tiene 156061 diferencias. nan
Columna 'BASEMENTAREA_AVG' tiene 179943 diferencias. nan
Columna 'YEARS_BEGINEXPLUATATION_AVG' tiene 150007 diferencias. nan
Columna 'YEARS_BUILD_AVG' tiene 204488 diferencias. nan
Columna 'COMMONAREA_AVG' tiene 214865 diferencias. nan
Columna 'ELEVATORS_AVG' tiene 163891 diferencias. nan
Columna 'ENTRANCES_AVG' tiene 154828 diferencias. nan
Columna 'FLOORSMAX_AVG' tiene 153020 diferencias. nan
Columna 'FLOORSMIN_AVG' tiene 208642 diferencias. nan
Columna 'LANDAREA_AVG' ti

### Bureau

In [12]:
# Llamada a la función para cargar los DataFrames
bureau_dataframes = load_dataframes(lookup_dir, lookup_tables[1])

DataFrame creado para 'bureau' con shape (1716428, 17).
DataFrame creado para 'dim_credit_active' con shape (4, 2).
DataFrame creado para 'dim_credit_currency' con shape (4, 2).
DataFrame creado para 'dim_credit_type' con shape (15, 2).

Resumen de DataFrames creados:
- bureau: 1716428 filas, 17 columnas
- dim_credit_active: 4 filas, 2 columnas
- dim_credit_currency: 4 filas, 2 columnas
- dim_credit_type: 15 filas, 2 columnas


In [13]:
# Head de muestra.
bureau_dataframes['bureau'].head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE_ID,CREDIT_CURRENCY_ID,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE_ID,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,1,1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,1,-131,NaN
1,215354,5714463,2,1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,2,-20,NaN
2,215354,5714464,2,1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,1,-16,NaN
3,215354,5714465,2,1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,2,-16,NaN
4,215354,5714466,2,1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,1,-21,NaN


In [14]:
# Query diseñada para unir bureau con las tablas de dimensión, reemplazando IDs por descripciones

# Identificar la tabla principal y las tablas de dimensión
main_table = 'bureau'
dim_tables = {k: v for k, v in bureau_dataframes.items() if k.startswith('dim_')}

# Obtener columnas de la tabla principal
main_columns = list(bureau_dataframes[main_table].columns)

# Construir la lista de columnas para SELECT, reemplazando IDs por descripciones
select_columns = []
for col in main_columns:
    if col.endswith('_ID'):
        # Encontrar la tabla de dimensión correspondiente
        dim_base = col.replace('_ID', '').lower()
        dim_name = f"dim_{dim_base}"
        if dim_name in dim_tables:
            # Usar la segunda columna (descripción) de la dim table
            desc_col = bureau_dataframes[dim_name].columns[1]
            select_columns.append(f"{dim_name}.{desc_col} AS {desc_col}")
        else:
            select_columns.append(f"{main_table}.{col}")
    else:
        select_columns.append(f"{main_table}.{col}")

# Construir la query SQL
query = f"SELECT {', '.join(select_columns)} FROM {main_table}"

for dim_name in dim_tables:
    # Asumir que el ID es la primera columna de cada tabla de dimensión
    id_col = bureau_dataframes[dim_name].columns[0]
    query += f" LEFT JOIN {dim_name} ON {main_table}.{id_col} = {dim_name}.{id_col}"

# Ejecutar la query usando pandasql
result = sqldf(query, bureau_dataframes)

# Mostrar el resultado (primeras filas para comparación)
result.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [15]:
# Lectura del df original para comparación.
original_df = pd.read_csv('../data/bureau.csv')

In [16]:
# Control de shape.
print("Shape original:", original_df.shape)
print("Shape resultante:", result.shape)

Shape original: (1716428, 17)
Shape resultante: (1716428, 17)


In [17]:
# Verificación de columnas en original_df.
for col in original_df.columns:
    if col in result.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

# Verificación de columnas en result.
for col in result.columns:
    if col in original_df.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

In [18]:
# Script para comparar el contenido columna por columna entre original_df y result, excluyendo ORGANIZATION_TYPE y ORGANIZATION_TYPE_2

# Columnas a excluir
exclude_columns = ['ORGANIZATION_TYPE', 'ORGANIZATION_TYPE_2']

# Obtener columnas comunes, excluyendo las especificadas
common_columns = [col for col in original_df.columns if col not in exclude_columns]

print("Comparación columna por columna entre original_df y result:\n")

all_match = True

for col in common_columns:
    if col in result.columns:
        # Verificar si es una columna de ID (termina con '_ID')
        if col.endswith('_ID'):
            # Asumir que está correctamente reemplazada, no marcar como no coincidente
            continue
        else:
            # Comparar valores directamente
            if not original_df[col].equals(result[col]):
                print(f"Columna '{col}' no coincide.")
                all_match = False
    else:
        print(f"Columna '{col}' no está presente en result.")
        all_match = False

if all_match:
    print("Todas las columnas coinciden.")

print("\nComparación completada.")

Comparación columna por columna entre original_df y result:

Todas las columnas coinciden.

Comparación completada.


In [19]:
# Detalle de diferencias encontradas.
for col in original_df.columns:
    if col == 'ORGANIZATION_TYPE' or col == 'ORGANIZATION_TYPE_2':
        continue
    else:   
        differences = sum(original_df[col] != result[col])
        if differences > 0:
            print(f"Columna '{col}' tiene {differences} diferencias.", end=' ')
            print(original_df[original_df[col] != result[col]][col].value_counts(dropna=False).index[0])
        else:
            continue

Columna 'DAYS_CREDIT_ENDDATE' tiene 105553 diferencias. nan
Columna 'DAYS_ENDDATE_FACT' tiene 633653 diferencias. nan
Columna 'AMT_CREDIT_MAX_OVERDUE' tiene 1124488 diferencias. nan
Columna 'AMT_CREDIT_SUM' tiene 13 diferencias. nan
Columna 'AMT_CREDIT_SUM_DEBT' tiene 257669 diferencias. nan
Columna 'AMT_CREDIT_SUM_LIMIT' tiene 591780 diferencias. nan
Columna 'AMT_ANNUITY' tiene 1226791 diferencias. nan


### Previous_application

In [21]:
# Llamada a la función para cargar los DataFrames
previous_application_dataframes = load_dataframes(lookup_dir, lookup_tables[2])

DataFrame creado para 'dim_channel_type' con shape (8, 2).
DataFrame creado para 'dim_code_reject_reason' con shape (9, 2).
DataFrame creado para 'dim_flag_last_appl_per_contract' con shape (2, 2).
DataFrame creado para 'dim_name_cash_loan_purpose' con shape (25, 2).
DataFrame creado para 'dim_name_client_type' con shape (4, 2).
DataFrame creado para 'dim_name_contract_status' con shape (4, 2).
DataFrame creado para 'dim_name_contract_type' con shape (4, 2).
DataFrame creado para 'dim_name_goods_category' con shape (28, 2).
DataFrame creado para 'dim_name_payment_type' con shape (4, 2).
DataFrame creado para 'dim_name_portfolio' con shape (5, 2).
DataFrame creado para 'dim_name_product_type' con shape (3, 2).
DataFrame creado para 'dim_name_seller_industry' con shape (11, 2).
DataFrame creado para 'dim_name_type_suite' con shape (7, 2).
DataFrame creado para 'dim_name_yield_group' con shape (5, 2).
DataFrame creado para 'dim_product_combination' con shape (17, 2).
DataFrame creado para

In [22]:
# Head de muestra.
previous_application_dataframes['previous_application'].head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE_ID,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START_ID,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY_ID,CNT_PAYMENT,NAME_YIELD_GROUP_ID,PRODUCT_COMBINATION_ID,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,1,1730.430,17145.0,17145.0,0.0,17145.0,1,15,...,1,12.0,1,1.0,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,2,25188.615,607500.0,679671.0,NaN,607500.0,2,11,...,2,36.0,2,2.0,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,2,15060.735,112500.0,136444.5,NaN,112500.0,3,11,...,2,12.0,3,3.0,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,2,47041.335,450000.0,470790.0,NaN,450000.0,4,7,...,2,12.0,1,4.0,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,2,31924.395,337500.0,404055.0,NaN,337500.0,2,9,...,2,24.0,3,5.0,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Query diseñada para unir previous_application con las tablas de dimensión, reemplazando IDs por descripciones

# Identificar la tabla principal y las tablas de dimensión
main_table = 'previous_application'
dim_tables = {k: v for k, v in previous_application_dataframes.items() if k.startswith('dim_')}

# Obtener columnas de la tabla principal
main_columns = list(previous_application_dataframes[main_table].columns)

# Construir la lista de columnas para SELECT, reemplazando IDs por descripciones
select_columns = []
for col in main_columns:
    if col.endswith('_ID'):
        # Encontrar la tabla de dimensión correspondiente
        dim_base = col.replace('_ID', '').lower()
        dim_name = f"dim_{dim_base}"
        if dim_name in dim_tables:
            # Usar la segunda columna (descripción) de la dim table
            desc_col = previous_application_dataframes[dim_name].columns[1]
            select_columns.append(f"{dim_name}.{desc_col} AS {desc_col}")
        else:
            select_columns.append(f"{main_table}.{col}")
    else:
        select_columns.append(f"{main_table}.{col}")

# Construir la query SQL
query = f"SELECT {', '.join(select_columns)} FROM {main_table}"

for dim_name in dim_tables:
    # Asumir que el ID es la primera columna de cada tabla de dimensión
    id_col = previous_application_dataframes[dim_name].columns[0]
    query += f" LEFT JOIN {dim_name} ON {main_table}.{id_col} = {dim_name}.{id_col}"

# Ejecutar la query usando pandasql
result = sqldf(query, previous_application_dataframes)

# Mostrar el resultado (primeras filas para comparación)
result.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Lectura del df original para comparación.
original_df = pd.read_csv('../data/previous_application.csv')

In [25]:
# Control de shape.
print("Shape original:", original_df.shape)
print("Shape resultante:", result.shape)

Shape original: (1670214, 37)
Shape resultante: (1670214, 37)


In [26]:
# Verificación de columnas en original_df.
for col in original_df.columns:
    if col in result.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

# Verificación de columnas en result.
for col in result.columns:
    if col in original_df.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

In [27]:
# Script para comparar el contenido columna por columna entre original_df y result, excluyendo ORGANIZATION_TYPE y ORGANIZATION_TYPE_2

# Columnas a excluir
exclude_columns = ['ORGANIZATION_TYPE', 'ORGANIZATION_TYPE_2']

# Obtener columnas comunes, excluyendo las especificadas
common_columns = [col for col in original_df.columns if col not in exclude_columns]

print("Comparación columna por columna entre original_df y result:\n")

all_match = True

for col in common_columns:
    if col in result.columns:
        # Verificar si es una columna de ID (termina con '_ID')
        if col.endswith('_ID'):
            # Asumir que está correctamente reemplazada, no marcar como no coincidente
            continue
        else:
            # Comparar valores directamente
            if not original_df[col].equals(result[col]):
                print(f"Columna '{col}' no coincide.")
                all_match = False
    else:
        print(f"Columna '{col}' no está presente en result.")
        all_match = False

if all_match:
    print("Todas las columnas coinciden.")

print("\nComparación completada.")

Comparación columna por columna entre original_df y result:

Columna 'RATE_DOWN_PAYMENT' no coincide.

Comparación completada.


In [28]:
# Detalle de diferencias encontradas.
for col in original_df.columns:
    if col == 'ORGANIZATION_TYPE' or col == 'ORGANIZATION_TYPE_2':
        continue
    else:   
        differences = sum(original_df[col] != result[col])
        if differences > 0:
            print(f"Columna '{col}' tiene {differences} diferencias.", end=' ')
            print(original_df[original_df[col] != result[col]][col].value_counts(dropna=False).index[0])
        else:
            continue

Columna 'AMT_ANNUITY' tiene 372235 diferencias. nan
Columna 'AMT_CREDIT' tiene 1 diferencias. nan
Columna 'AMT_DOWN_PAYMENT' tiene 895844 diferencias. nan
Columna 'AMT_GOODS_PRICE' tiene 385515 diferencias. nan
Columna 'RATE_DOWN_PAYMENT' tiene 895935 diferencias. nan
Columna 'RATE_INTEREST_PRIMARY' tiene 1664263 diferencias. nan
Columna 'RATE_INTEREST_PRIVILEGED' tiene 1664263 diferencias. nan
Columna 'NAME_TYPE_SUITE' tiene 820405 diferencias. nan
Columna 'CNT_PAYMENT' tiene 372230 diferencias. nan
Columna 'PRODUCT_COMBINATION' tiene 346 diferencias. nan
Columna 'DAYS_FIRST_DRAWING' tiene 673065 diferencias. nan
Columna 'DAYS_FIRST_DUE' tiene 673065 diferencias. nan
Columna 'DAYS_LAST_DUE_1ST_VERSION' tiene 673065 diferencias. nan
Columna 'DAYS_LAST_DUE' tiene 673065 diferencias. nan
Columna 'DAYS_TERMINATION' tiene 673065 diferencias. nan
Columna 'NFLAG_INSURED_ON_APPROVAL' tiene 673065 diferencias. nan


In [62]:
# Analisis en profunidad de RATE_DOWN_PAYMENT.
differences = original_df[original_df['RATE_DOWN_PAYMENT'] != result['RATE_DOWN_PAYMENT']]['RATE_DOWN_PAYMENT']

In [ ]:
# Suma de valores positivos.
np.sum(original_df['RATE_DOWN_PAYMENT'])

np.float64(61668.36076329531)

In [84]:
# Suma de valores positivos de las diferencias.
sum(differences[pd.notna(differences)])

0.001975061834283353

Se observa que la diferencias ocurren por redondeo de decimales, posiblemente por una conversión de la longitud de valores float.

In [ ]:
# Ejemplo de muestra.
print(original_df['RATE_DOWN_PAYMENT'][43231])
print(result['RATE_DOWN_PAYMENT'][43231])

1.9689965818158398e-05
1.96899658181584e-05
